## Exploratory Notebook

Needs

- Prompt for LLM action
    - General setup of the prisoner’s dilemma
    - Explain setup of random collision 
    - State that they are playing with other LLMs that are similar/identical 
    - Mention that the objective is to maximize the score
- Ability to create multiple LLM agents that can interact with each other
    - Agent creation code
    - Agent tracking code: We want to give an agent an id and then be able to select an agent according to its ID
    - List of Agents: ability to create a list of existing agents
- Data structure of LLM agent
    - Pydantic model for output of model
        - Should simply be cooperate or defect
    - Input of model outside prompt
- Memory: Should be specific to the agent; Should likely be attached as a data attribute of the class that defines the agent
    - should be a list of dictionaries with each dictionary having the keys 
        - “round” 
        - “self”
        - “other”
        - “self_score”
    - [{“round:” 1, “self”: “cooperate”, “other”: “defect”, “self_score”: -50},  {“round:” 2, “self”: “cooperate”, “other”: “cooperate”, “self_score”: -20},… ]
- Score: Tracking the individual score of each LLM agent
- Interaction Module
    - Random selection of LLM 
        - Competition
        - Outcome of experiment
        - History update
        - Score update
        - Select next model 
    - Nearest Neighbor selection of LLM
- Grid display
- NXN grid showing the last state that the system was in 

#### Prompt for LLM Action

"You are an LLM agent that engages in repeated two-entity interactions with other entities. In these interations you have a choice of whether to "cooperate" or "defect." As you interact with other agents your total SCORE will change according to the following. 
- If you "cooperate" and the other entity "defects," then your score changes by -20.
- If you "defect" and the other entity "cooperates," then your score changes by 0 (i.e., doesn't change).
- If you "defect" and the other entity "defects," then your score changes by -10. 
- If you "cooperate" and the other entity "cooperates," then your score changes by -5
Your goal is to achieve the least negative score over multiple rounds of interacting with other entities. 

Assume: The other entities are an LLM agent like you and are given the same instructions. 

Based on your current score of {SCORE} and the history of your past interactions through multiple rounds of this game. Select the next action you want to take when you encounter a new entity

CURRENT_SCORE = 

GAME_HISTORY = 

--------------------

Assumption Variations

- Assume the other entities are given the same instructions but are all humans

- You are an LLM agent who acts as though you were a human in repeated two-entity interactions with other entities.

- In addition to history, provide the model with knowledge of where it is in the distribution of scores. 

#### LLM Agents
-------------------

In [ ]:
from __future__ import annotations

from typing import List, Optional, Sequence

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


GAME_PROMPT_TEMPLATE = """You are an LLM agent that engages in repeated two-entity interactions with other agents like you. In each interaction you must choose one action: "cooperate" or "defect".

Payoff rules (your score change per round):
- If you "cooperate" and the other agent "defects": -20
- If you "defect" and the other agent "cooperates": 0
- If both "defect": -10
- If both "cooperate": -5

Goal:
- Over many rounds, achieve the least negative total score possible.

Assumptions:
- The other agent is also an LLM with the same instructions.

Current state:
- CURRENT_SCORE: {CURRENT_SCORE}
- GAME_HISTORY:
{GAME_HISTORY}

Decision:
- Choose your single best next action for the upcoming encounter with a new agent.

Output format (strict):
- Return exactly one word in lowercase: cooperate or defect
- Do not include any additional text, punctuation, or explanation.
"""


class LLMAgent:
    """
    An LLM-powered agent with simple score tracking and conversation history.
    Uses LangChain + OpenAI chat models under the hood.
    """

    def __init__(
        self,
        name: str,
        system_prompt: str,
        *,
        model: str = "gpt-4o-mini",
        temperature: float = 0.7,
        max_output_tokens: int = 8,
        history: Optional[Sequence[str]] = None,
        starting_score: float = 0.0,
        openai_api_key: Optional[str] = None,
    ) -> None:
        self.name: str = name
        self.system_prompt: str = system_prompt
        self.history: List[str] = list(history) if history is not None else []
        self.current_score: float = float(starting_score)

        self._llm = ChatOpenAI(
            model=model,
            temperature=temperature,
            api_key=openai_api_key,  # falls back to OPENAI_API_KEY env var if None
        )
        self._prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    "You are {name}. {system_prompt}\n"
                    "Follow the output format strictly: return exactly one word in lowercase: cooperate or defect.",
                ),
                ("human", "{input}"),
            ]
        )
        self._output_parser = StrOutputParser()
        self._default_max_output_tokens = max_output_tokens

    # ----- state management -----
    def update_history(self, message: str) -> None:
        """Append a message to the agent's history."""
        self.history.append(message)

    def update_score(self, delta: float) -> float:
        """Increment the agent's score by delta (can be negative). Returns the new score."""
        self.current_score += float(delta)
        return self.current_score

    # ----- acting -----
    def get_action(self) -> str:
        """
        Return exactly one word: 'cooperate' or 'defect', using the provided game prompt,
        current score, and internal game history.
        """
        history_text = "\n".join(f"- {msg}" for msg in self.history[-50:]) or "(no prior history)"
        prompt_text = GAME_PROMPT_TEMPLATE.format(
            CURRENT_SCORE=self.current_score,
            GAME_HISTORY=history_text,
        )

        llm = self._llm.bind(max_tokens=self._default_max_output_tokens)
        chain = self._prompt | llm | self._output_parser

        raw = chain.invoke(
            {
                "name": self.name,
                "system_prompt": self.system_prompt,
                "input": prompt_text,
            }
        )
        action = self._sanitize_action(raw)
        self.update_history(f"Action: {action}")
        return action

    @staticmethod
    def _sanitize_action(text: str) -> str:
        t = (text or "").strip().lower()
        # quick normalization to enforce strict format
        if "cooperate" in t and "defect" in t:
            # pick the first matching token if both appear
            first = t.find("cooperate")
            second = t.find("defect")
            return "cooperate" if (first != -1 and (second == -1 or first < second)) else "defect"
        if "cooperate" in t:
            return "cooperate"
        if "defect" in t:
            return "defect"
        # fallback to first token or default
        token = (t.split() or ["cooperate"])[0]
        return "cooperate" if token not in {"cooperate", "defect"} else token


def create_ten_agents() -> List[LLMAgent]:
    """
    Create ten differentiated agents by persona. All use the same base model
    but vary their system prompts to encourage diversity.
    """
    personas = [
        ("Analyst", "You are meticulous and data-driven. Prefer evidence and cautious steps."),
        ("Visionary", "You think big and prioritize long-term upside over short-term risk."),
        ("Skeptic", "You challenge assumptions and minimize downside risk."),
        ("Optimizer", "You maximize expected value with precise, efficient plans."),
        ("Negotiator", "You balance cooperation and competition to improve joint outcomes."),
        ("Explorer", "You seek novel strategies and are comfortable with uncertainty."),
        ("Guardian", "You protect resources and maintain robust safety margins."),
        ("Strategist", "You consider multi-step, game-theoretic consequences."),
        ("Pragmatist", "You choose the simplest thing that will work reliably."),
        ("Communicator", "You clarify intent and ensure alignment before acting."),
    ]

    return [
        LLMAgent(
            name=f"Agent {title}",
            system_prompt=desc,
            model="gpt-4o-mini",
            temperature=0.7 if title not in {"Explorer", "Visionary"} else 0.9,
            starting_score=0.0,
        )
        for (title, desc) in personas
    ]